# 04 · Testing

Four kinds of tests, all of which you will run (and break) here:

| kind | defined | example in this repo |
|---|---|---|
| generic (built-in) | YAML under a column | `unique`, `not_null`, `accepted_values`, `relationships` |
| generic (custom) | `tests/generic/*.sql` | `positive_value` |
| generic (package) | dbt_utils / dbt_expectations | `accepted_range`, `expect_column_values_to_be_between` |
| singular | `tests/*.sql` | `assert_order_payments_reconcile` |
| **unit** (1.8+) | YAML `unit_tests:` | `unit_int_order_payments_pivot` |

Every data test is a SELECT returning **bad rows**; zero rows = pass.

Companion reading: [docs/07](../docs/07_testing.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. Green baseline

In [ ]:
res = run_dbt(["test"])
assert res.success

## 2. Break a test on purpose

Inject an order with an impossible status into the RAW table (upstream of
dbt), then re-test just `stg_orders`. `dbtRunner` returns a result object
instead of exiting, so a failing run is something we can *inspect*.

In [ ]:
with engine.begin() as c:
    c.execute(text("""
        insert into raw.raw_orders values
        (999999, 1, 'teleported', now(), now(), now(), 999)
    """))

res = run_dbt(["test", "--select", "stg_orders"])
assert not res.success, "expected the accepted_values test to fail"

for r in res.result.results:
    print(f"{r.status:<6} {r.node.name}")

## 3. `store_failures`: inspect WHICH rows failed

The `accepted_values` test on `stg_orders.status` is configured with
`store_failures: true`, so dbt materialized the offending rows into an
audit schema. This is how you debug a red test without re-running its SQL
by hand.

In [ ]:
# The stored-failures relation gets a generated (possibly truncated) name --
# discover it instead of hardcoding. (Deliberately no LIKE '%...' here: a
# literal % in raw SQL through the psycopg2 driver reads as a parameter
# placeholder. Use a regex match, or sqlalchemy.text(), instead.)
rels = pd.read_sql("""
    select table_schema || '.' || table_name as rel
    from information_schema.tables
    where table_schema ~ 'dbt_test__audit$'
""", engine)["rel"].tolist()
print(rels)

failures_rel = next(r for r in rels if "accepted_values_stg_orders" in r)
pd.read_sql(f"select * from {failures_rel}", engine)

## 4. Severity: `warn` nags, but does not fail the build

In [ ]:
with engine.begin() as c:
    c.execute(text("""
        insert into raw.raw_events values
        (99999999, 1, 'hologram_view', '/holodeck', now(), now(), 999)
    """))

res = run_dbt(["test", "--select", "stg_events"])
assert res.success, "warn severity must NOT fail the run"

for r in res.result.results:
    print(f"{r.status:<6} {r.node.name}")

The `accepted_values` test on `event_type` has `severity: warn`: a new event
type appearing upstream is worth knowing about, not worth blocking the
nightly build over.

## 5. Cleanup -- remove the injected rows, prove we're green again

In [ ]:
with engine.begin() as c:
    c.execute(text("delete from raw.raw_orders where id = 999999"))
    c.execute(text("delete from raw.raw_events where id = 99999999"))

res = run_dbt(["test", "--select", "stg_orders", "stg_events"])
assert res.success

## 6. Unit tests: test the LOGIC, not the data

Data tests validate what's in the warehouse *after* building. The unit test
`unit_int_order_payments_pivot` instead feeds **mocked input rows** through
the model's SQL and compares against expected output -- it would catch a
broken pivot even on an empty warehouse. In `dbt build`, unit tests run
*before* the model builds and gate it.

In [ ]:
res = run_dbt(["test", "--select", "test_type:unit"])
assert res.success

Open `models/intermediate/_intermediate__models.yml` to see the fixture
format -- including two subtle bits this repo already hit for you: macro
overrides (`get_column_values` cannot introspect a mocked CTE) and quoting
decimal expectations so YAML doesn't eat trailing zeros.